# Keras Hub を使用した転移学習とファインチューニング
このノートブックでは、[Keras Hub](https://keras.io/keras_hub/) の事前学習済みモデルを使用した**転移学習（transfer learning）**と**ファインチューニング（fine-tuning）**によって、モデルをスクラッチから学習するよりも高い画像分類精度を達成する方法を実践します。これは、データセットが小さい場合、計算リソースが限られている場合、あるいは大規模モデルをゼロから学習する時間がない場合に、一般的かつ効果的な戦略です。

**学習目標**
- **転移学習**: 特徴抽出器として事前学習済みモデルを活用する方法を学ぶ
- **ファインチューニング**: 特定のタスクに合わせて事前学習済みモデルの重みをファインチューニングする方法を理解する
- **学習率スケジューリング**: 安定して効果的なファインチューニングを達成するために学習率スケジュールをどのように利用できるかを理解する

## セットアップ

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import pathlib

import IPython.display as display
import keras
import keras_hub
import matplotlib.pylab as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Sequential
from keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
    Softmax,
)
from PIL import Image

## データの探索

通常通り、モデルの構築を始める前にデータを確認しましょう。ここでは、クリエイティブ・コモンズ・ライセンスの 3,670 枚の花の写真データセットを使用します。画像は 'daisy'（デイジー）、'roses'（バラ）、'dandelion'（タンポポ）、'sunflowers'（ヒマワリ）、'tulips'（チューリップ）の 5 つのカテゴリに分類されています。

以下の [keras.utils.get_file](https://keras.io/api/utils/python_utils/#getfile-function) コマンドにより、ローカルの Keras キャッシュにデータセットがダウンロードされます。ターミナルからファイルを確認するには、下のセルの出力をコピーしてください。

In [ ]:
data_dir = keras.utils.get_file(
    "flower_photos",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
    untar=True,
)

# データパスを出力
print("cd", data_dir)


Python の組み込みツールである [pathlib](https://docs.python.org/3/library/pathlib.html) を使用して、この非構造化データを把握できます。

In [ ]:
data_dir_path = pathlib.Path(data_dir)

image_count = len(list(data_dir_path.glob("*/*/*.jpg")))
print("There are", image_count, "images.")

CLASS_NAMES = np.array(
    [
        item.name
        for item in data_dir_path.glob("*/*")
        if item.name != "LICENSE.txt"
    ]
)
print("These are the available classes:", CLASS_NAMES)

モデルが何を学習しようとしているのかを確認するために、いくつかの画像を表示してみましょう。

In [ ]:
roses = list(data_dir_path.glob("*/roses/*"))

for image_path in roses[:3]:
    display.display(Image.open(str(image_path)))

## データセットの構築
機械学習モデルを学習させるには、データを整理して効率的に読み込む必要があります。ここでは [keras.utils.image_dataset_from_directory](https://keras.io/api/data_loading/image/#imagedatasetfromdirectory-function) を使用して tf.data.Dataset オブジェクトを作成します。

このメソッドは、データが次のようなディレクトリ構造で保存されていることを前提としています:



```
main_directory/
...class_a/
......a_image_1.jpg
......a_image_2.jpg
...class_b/
......b_image_1.jpg
......b_image_2.jpg
```

このユーティリティは次の処理を自動的に処理します:
- ファイルの検出: ディレクトリ構造をスキャンしてすべての画像ファイルを検出します。
- ラベル付け: サブディレクトリ名（例: roses, tulips）からクラスラベルを推論します。
- 分割: `validation_split` の指定に従ってデータをトレーニング用（80%）と検証用（20%）のサブセットに分割し、両方のサブセットを返します（`subset="both"`）。
- リサイズ: 後で使用するモデルアーキテクチャと互換性を持たせるため、すべての画像を一定のサイズ（224x224 ピクセル）にリサイズします。
- バッチ処理: 効率的な学習のために画像をバッチ（バッチサイズ 32）にまとめます。

このプロセスにより、手動のループや複雑なファイル処理を行うことなく、モデルが利用できる状態のデータを準備できます。

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
# 10 はこのデータセットのローカル学習用に調整されたマジックナンバーです。
VALIDATION_SPLIT = 0.2

SEED = 999


In [ ]:
train_ds, eval_ds = keras.utils.image_dataset_from_directory(
    f"{data_dir}/flower_photos",
    seed=SEED,
    validation_split=VALIDATION_SPLIT,
    subset="both",
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    label_mode="categorical",
)

## シンプルな CNN モデル

これらの花の写真は、MNIST の手書き数字認識画像よりもはるかに大きいです。軸あたりのピクセル数は約 10 倍であり、**さらに** 3 つのカラーチャンネルがあるため、情報量は 200 倍以上になります。

また、ピクセル範囲を [0,255] から [0,1] にリスケールするための `Rescaling` レイヤを追加しました。この処理はデータパイプラインでも適用できますが、モデル自体に組み込むことでトレーニングサービングスキュー（training-serving skew）を防ぐことができます。

これまでの手法はどれほど通用するでしょうか？<a href="2_mnist_cnn.ipynb">MNIST models lab</a> からベストなモデルアーキテクチャをコピーし、10 エポック学習させた後のパフォーマンスを確認してみましょう。

In [ ]:
nclasses = len(CLASS_NAMES)
hidden_layer_1_neurons = 400
hidden_layer_2_neurons = 100
dropout_rate = 0.25
num_filters_1 = 64
kernel_size_1 = 3
pooling_size_1 = 2
num_filters_2 = 32
kernel_size_2 = 3
pooling_size_2 = 2

layers = [
    keras.layers.Rescaling(scale=1.0 / 255),
    Conv2D(
        num_filters_1,
        kernel_size=kernel_size_1,
        activation="relu",
        input_shape=(IMG_WIDTH, IMG_HEIGHT, 3),
    ),
    MaxPooling2D(pooling_size_1),
    Conv2D(num_filters_2, kernel_size=kernel_size_2, activation="relu"),
    MaxPooling2D(pooling_size_2),
    Flatten(),
    Dense(hidden_layer_1_neurons, activation="relu"),
    Dense(hidden_layer_2_neurons, activation="relu"),
    Dropout(dropout_rate),
    Dense(nclasses),
    Softmax(),
]

cnn_model = Sequential(layers)
cnn_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
history = cnn_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

## Keras Hub による転移学習

作成したモデルを実行すると、ランダムよりはわずかに良い程度で学習は進むものの、過学習（overfitting）を起こしてしまいます。今回のデータセットに対してモデルが浅すぎるため、低レベルな特徴（色など）ばかりを利用してしまい、画像全体のセマンティクスを捉えられていない可能性があります。

シンプルな CNN の限界を克服するために、転移学習を導入します。

ここでは、Keras Hub の事前学習済み画像モデルである MobileNetV3 を活用し、今回のタスク向けに再利用します。このモデルは軽量でモバイル推論向けに構築されているため理想的です。

`keras_hub.models.Backbone.from_preset()` 関数でモデル識別子を指定することにより、事前学習済みの特徴抽出器オブジェクトを取得できます。利用可能なモデルについては[こちらのドキュメント](https://keras.io/keras_hub/presets/)を参照してください。

ここでは転移学習のために `backbone.trainable` を `False` に設定し、事前学習済みモデルの膨大な重みが学習中に更新されないようにします。

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)
backbone.trainable = False

backbone.summary()

バックボーンの "Trainable params"（学習可能なパラメータ数）が 0 になっていることに注目してください。

代わりに、その上に新しい分類ヘッド（シンプルな Dense レイヤ）が追加され、花の画像を分類するためにこの新しいヘッドのみが学習されます。

このアプローチでは、バックボーンの強力な事前学習済み特徴量を活用することで、小規模データセットでの過学習のリスクを抑えつつ、高いパフォーマンスのベースラインを迅速に達成できます。

In [ ]:
transfer_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            nclasses,
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)
# transfer_model.build((None,) + (IMG_HEIGHT, IMG_WIDTH, 3))
transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

In [ ]:
history = transfer_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

よくなりましたね。それでも、まだ改善の余地があります。

## MobileNet のファインチューニング
転移学習のステップに続いて、ファインチューニングがどのように機能するかを見てみましょう。ここではバックボーンの凍結を解除し、分類ヘッドと一緒に学習させます。

学習可能な状態を保つため、`backbone.trainable` を `False` に設定**しない**ことに注意してください。

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

# ファインチューニングのために trainable を True のままにします
# backbone.trainable = False

finetune_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            nclasses,
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)
finetune_model.summary()


In [ ]:
finetune_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

history = finetune_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

トレーニング精度は先ほどの場合より向上しているように見えますが、検証精度はほぼ横ばいのままです。

これは、MobileNet の多数のパラメータを調整するようになったため、この小さなデータセットに対して容易に過学習してしまうからです。

さらに、デフォルトの学習率によって、MobileNet が事前学習中に獲得した知識が損なわれてしまった（破壊的忘却が起きた）可能性もあります。効果的なファインチューニングには、学習率の慎重な選択が不可欠です。

## ファインチューニングのための学習率スケジュール

それでは、ファインチューニングのための追加テクニックを取り入れましょう。
ファインチューニングは学習率に非常に敏感であるため、慎重に設計された**学習率スケジュール**を使用することが重要です。


一般的な学習率スケジュールには、主に 2 つのフェーズがあります:
- **ウォームアップ フェーズ（Warmup Phase）**: 事前学習済みモデルのパターンが壊れるのを防ぎ、新しいタスクに徐々に適応させるため、学習率を小さく開始して徐々に増加させます
- **減衰フェーズ（Decay Phase）**: よりきめ細かい調整と良好な収束を実現するために、学習率を徐々に減少させます

ここでは、ウォームアップ期間を柔軟に指定できる `keras.optimizers.schedules.CosineDecay` を使用してこのスケジュールを設計します。

In [ ]:
epochs = 10
steps_per_epoch = tf.data.experimental.cardinality(train_ds).numpy()
num_train_steps = steps_per_epoch * epochs
num_warmup_steps = int(0.1 * num_train_steps)

warmup_tgt = 0.001
scheduler = keras.optimizers.schedules.CosineDecay(
    0.0,
    num_train_steps,
    warmup_target=warmup_tgt,
    warmup_steps=num_warmup_steps,
)

optimizer = keras.optimizers.Adam(learning_rate=scheduler)

In [ ]:
plt.plot([scheduler(lr) for lr in range(num_train_steps)])
plt.title("Learning Rate Schedule")
plt.xlabel("Steps")
plt.ylabel("Learning Rate")
plt.show()

In [ ]:
backbone = keras_hub.models.Backbone.from_preset(
    "mobilenet_v3_large_100_imagenet_21k",
)

ft_lr_schedule_model = keras.Sequential(
    [
        keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        keras.layers.Rescaling(scale=1.0 / 255),
        backbone,
        keras.layers.GlobalMaxPooling2D(),
        keras.layers.Dropout(rate=0.2),
        keras.layers.Dense(
            nclasses,
            activation="softmax",
            kernel_regularizer=keras.regularizers.l2(0.0001),
        ),
    ]
)

新しい学習率スケジューラを使用して同じモデルを学習させてみましょう。

In [ ]:
ft_lr_schedule_model.compile(
    optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"]
)

history = ft_lr_schedule_model.fit(
    train_ds,
    epochs=10,
    validation_data=eval_ds,
)

In [ ]:
ACCURACY_COLS = ["accuracy", "val_accuracy"]
_ = pd.DataFrame(history.history)[ACCURACY_COLS].plot()

新しい学習率スケジュールのおかげで、より安定した学習曲線が得られ、トレーニングと検証の両方でより高いパフォーマンスが確認できるようになりました。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.